# Loan Approval Classification — Assignment Five (Part B2)

This notebook trains **Logistic Regression** and **Random Forest** models following the Lesson 5 workflow. It also includes the required third classifier, **K-Nearest Neighbors (KNN)**, for comparison.

The models use the cleaned loan dataset with engineered features, following the same preprocessing steps and evaluation structure.

In [30]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier  # third classifier (assignment requirement)
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


RANDOM_STATE = 42

print("Successfully imported libraries.")

Successfully imported libraries.


## STEP 1 - Load the cleaned dataset

In [31]:
CSV_PATH = "clean_loan_dataset.csv"

df = pd.read_csv(CSV_PATH)
df.head()

,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved,DebtToIncome,IncomePerYearEmployed
0,0.822149,-0.653722,-0.978632,0.246080,1,0,0,-0.445244,8.274536
1,0.564574,-0.737864,0.209402,-0.458384,1,0,1,-0.912749,0.153994
2,-0.856268,0.000000,-0.611111,-0.414958,0,0,1,0.280113,-0.126321
3,-0.911156,-0.498382,0.431624,-0.661037,1,0,1,-0.315175,-0.669033
4,-0.649046,-0.718447,-0.235043,-0.482509,0,0,1,-0.284688,-0.284975


## STEP 2 - Split features (X) and target (y)
Features = applicant info; label = Approved (1) or Rejected (0)

In [32]:
X = df.drop(columns=["Approved"])
y = df["Approved"]

## STEP 3 - Train/test split (stratified)
L2: stratify keeps class ratio in both sets -- important for classification


In [33]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("=== SPLIT SIZES ===")
print(f"Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}")


=== SPLIT SIZES ===
Train: 80  |  Test: 20


## STEP 4 - Train Classifiers

This notebook trains **Logistic Regression** and **Random Forest** using the same settings as the Lesson 5 example. It also adds **K-Nearest Neighbors (KNN)** as the required third classifier.

KNN predicts a loan application's outcome by comparing it with the **five most similar** applicants in the training data. Because KNN relies on distances between features, scaling the data in Part B1 is important for better performance.

**Source:** scikit-learn documentation — `sklearn.neighbors.KNeighborsClassifier`.

In [34]:
log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
rf = RandomForestClassifier(n_estimators=1000, random_state=RANDOM_STATE)
knn = KNeighborsClassifier(n_neighbors=5)

log_reg.fit(X_train, y_train)
rf.fit(X_train, y_train)
knn.fit(X_train, y_train)

print("All three models trained.")


All three models trained.


## STEP 5 - Helper functions: metrics + confusion matrix

In [35]:
def print_clf_metrics(name, y_true, y_pred, pos_label=1):
    """Print Accuracy, Precision, Recall, F1. pos_label=1 means Approved is positive."""
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, pos_label=pos_label)
    rec = recall_score(y_true, y_pred, pos_label=pos_label)
    f1 = f1_score(y_true, y_pred, pos_label=pos_label)
    print(f"\n{name} Performance:")
    print(f"  Accuracy : {acc:.3f}")
    print(f"  Precision: {prec:.3f}  (positive = Approved=1)")
    print(f"  Recall   : {rec:.3f}  (positive = Approved=1)")
    print(f"  F1-Score : {f1:.3f}  (positive = Approved=1)")


def print_confmat(name, y_true, y_pred):
    """
    Confusion matrix with readable labels.
    Rows = Actual, Cols = Predicted
    Order: [Approved(1), Rejected(0)]
    """
    cm = confusion_matrix(y_true, y_pred, labels=[1, 0])
    cm_df = pd.DataFrame(
        cm,
        index=["Actual: Approved (1)", "Actual: Rejected (0)"],
        columns=["Pred: Approved (1)", "Pred: Rejected (0)"],
    )
    print(f"\n{name} - Confusion Matrix:\n{cm_df}")


def label_str(v):
    return "Approved (1)" if v == 1 else "Rejected (0)"


## STEP 6 - Show results for all three models

In [36]:
log_reg_preds = log_reg.predict(X_test)
print_clf_metrics("Logistic Regression", y_test, log_reg_preds, pos_label=1)
print_confmat("Logistic Regression", y_test, log_reg_preds)

rf_preds = rf.predict(X_test)
print_clf_metrics("Random Forest", y_test, rf_preds, pos_label=1)
print_confmat("Random Forest", y_test, rf_preds)

knn_preds = knn.predict(X_test)
print_clf_metrics("KNN (k=5)", y_test, knn_preds, pos_label=1)
print_confmat("KNN (k=5)", y_test, knn_preds)



Logistic Regression Performance:
  Accuracy : 0.700
  Precision: 0.733  (positive = Approved=1)
  Recall   : 0.846  (positive = Approved=1)
  F1-Score : 0.786  (positive = Approved=1)

Logistic Regression - Confusion Matrix:
                      Pred: Approved (1)  Pred: Rejected (0)
Actual: Approved (1)                  11                   2
Actual: Rejected (0)                   4                   3

Random Forest Performance:
  Accuracy : 0.700
  Precision: 0.733  (positive = Approved=1)
  Recall   : 0.846  (positive = Approved=1)
  F1-Score : 0.786  (positive = Approved=1)

Random Forest - Confusion Matrix:
                      Pred: Approved (1)  Pred: Rejected (0)
Actual: Approved (1)                  11                   2
Actual: Rejected (0)                   4                   3

KNN (k=5) Performance:
  Accuracy : 0.650
  Precision: 0.688  (positive = Approved=1)
  Recall   : 0.846  (positive = Approved=1)
  F1-Score : 0.759  (positive = Approved=1)

KNN (k=5) - Confus

## STEP 7 - Single-row sanity check

In [37]:
i = 2
x_one_df = X_test.iloc[[i]]
y_true = y_test.iloc[i]

print("\n=== SINGLE APPLICATION CHECK ===")
print(f"  Actual label : {label_str(y_true)}")
print(f"  Logistic Regression   : {label_str(int(log_reg.predict(x_one_df)[0]))}")
print(f"  Random Forest         : {label_str(int(rf.predict(x_one_df)[0]))}")
print(f"  KNN (k=5)             : {label_str(int(knn.predict(x_one_df)[0]))}")



=== SINGLE APPLICATION CHECK ===
  Actual label : Approved (1)
  Logistic Regression   : Approved (1)
  Random Forest         : Approved (1)
  KNN (k=5)             : Approved (1)
